## 04a_feature_engineering — RCM Intelligence Platform
### Purpose
Engineers the ML feature set for denial risk prediction.
Reads from Gold and Silver tables, creates a clean feature
matrix ready for MLflow model training.

### What this does
1. Reads fact_claims and hospital_scorecard from Gold
2. Engineers categorical and numerical features
3. Encodes categoricals with StringIndexer
4. Assembles feature vector
5. Writes ML-ready dataset to rcm_ml schema
6. Logs feature metadata to audit table

### Features engineered
### Numerical
- avg_submitted_charge, avg_total_payment, avg_medicare_payment
- charge_to_payment_ratio, revenue_gap_pct, medicare_payment_pct
- patient_responsibility, total_discharges, rcm_health_score

### Categorical (encoded)
- provider_state, hospital_type, hospital_ownership
- drg_severity_tier, rural_urban_classification

### Target variable
- denial_proxy: 1 if underpayment_flag=1 AND ctp_ratio > 75th percentile

### Outputs
- rcm_platform.rcm_ml.features_denial_risk

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | ML |
| Run order  | 8 — after 03b_kpi_aggregations |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA + ML TABLE
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, expr,
    percentile_approx, avg, stddev
)

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "04a_feature_engineering"

# ML feature table
TBL_ML_FEATURES = f"{ML_DB}.features_denial_risk"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"ML features tbl : {TBL_ML_FEATURES}")

In [0]:
# ============================================================
# STEP 1 — READ SOURCE TABLES
# ============================================================

print("=" * 55)
print("  READING SOURCE TABLES")
print("=" * 55)

df_fact      = spark.table(TBL_FACT_CLAIMS)
df_scorecard = spark.table(TBL_GOLD_SCORECARD)

print(f"fact_claims       : {df_fact.count():,} rows")
print(f"hospital_scorecard: {df_scorecard.count():,} rows")

# Get the 75th percentile of CTP ratio
# Used to define the denial proxy target variable
ctp_75th = df_fact.select(
    percentile_approx("charge_to_payment_ratio", 0.75)
    .alias("p75")
).collect()[0]["p75"]

# Force to Python float — prevents Spark type issues
ctp_75th = float(ctp_75th)

print(f"\nCTP ratio 75th percentile : {ctp_75th:.4f}")
print("Claims above this + underpayment flag = denial proxy = 1")

In [0]:
# ============================================================
# STEP 2 — BUILD FEATURE MATRIX
# Join fact to scorecard, engineer target variable
# ============================================================

print("=" * 55)
print("  BUILDING FEATURE MATRIX")
print("=" * 55)

# Get hospital-level RCM score from scorecard
df_hosp_score = df_scorecard.select(
    col("provider_id"),
    col("rcm_health_score"),
    col("rcm_health_grade"),
    col("avg_ctp_ratio").alias("hospital_avg_ctp"),
    col("underpayment_rate_pct").alias("hospital_underpayment_rate")
)

# Join and build features
df_features = df_fact \
    .join(df_hosp_score, on="provider_id", how="left") \
    .select(
        # Identifiers
        col("provider_id"),
        col("drg_code"),
        col("provider_state"),

        # Numerical features
        col("avg_submitted_charge"),
        col("avg_total_payment"),
        col("avg_medicare_payment"),
        col("charge_to_payment_ratio"),
        col("revenue_gap_pct"),
        col("medicare_payment_pct"),
        col("patient_responsibility"),
        col("total_discharges"),
        col("rcm_health_score"),
        col("hospital_avg_ctp"),
        col("hospital_underpayment_rate"),

        # Categorical features
        col("hospital_type"),
        col("hospital_ownership"),
        col("drg_severity_tier"),
        col("rural_urban_classification"),
        col("emergency_services"),

        # Raw flags
        col("underpayment_flag"),
        col("high_volume_flag")
    ) \
    .withColumn("denial_proxy",
        # Broader target — captures more denial risk patterns
        # 1 = ANY of these high-risk conditions:
        #   - Underpaid AND high CTP ratio (original)
        #   - Extremely high CTP ratio (>8x) regardless of underpayment
        #   - Revenue gap > 85% (hospital collecting < 15 cents per dollar)
        when(
            ((col("underpayment_flag") == 1) &
             (col("charge_to_payment_ratio") > lit(float(ctp_75th))))
            |
            (col("charge_to_payment_ratio") > lit(8.0))
            |
            (col("revenue_gap_pct") > lit(85.0)),
            lit(1)
        ).otherwise(lit(0))
    ) \
    .withColumn("_feature_batch_id", lit(BATCH_ID)) \
    .withColumn("_feature_date",     lit(BATCH_DATE)) \
    .dropna(subset=[
        "avg_submitted_charge",
        "avg_total_payment",
        "charge_to_payment_ratio",
        "revenue_gap_pct"
    ])

feature_count = df_features.count()
denial_count  = df_features.filter("denial_proxy = 1").count()
denial_rate   = float(denial_count) / float(feature_count) * 100

print(f"Total rows       : {feature_count:,}")
print(f"Denial proxy = 1 : {denial_count:,} ({denial_rate:.2f}%)")
print(f"Denial proxy = 0 : {feature_count - denial_count:,} ({100 - denial_rate:.2f}%)")
print(f"\nClass balance — ideal is 20-40% positive rate")
print(f"Our denial proxy rate: {denial_rate:.2f}%")

In [0]:
# ============================================================
# STEP 3 — ENCODE CATEGORICALS WITH SPARK SQL
# Using SQL-based encoding instead of ML Pipeline
# More memory efficient for Serverless compute
# ============================================================

print("=" * 55)
print("  ENCODING CATEGORICALS + WRITING FEATURES")
print("=" * 55)

# Fill nulls in categoricals
df_filled = df_features \
    .fillna("Unknown", subset=[
        "hospital_type",
        "hospital_ownership",
        "drg_severity_tier",
        "rural_urban_classification",
        "emergency_services"
    ]) \
    .fillna(0.0, subset=[
        "rcm_health_score",
        "hospital_avg_ctp",
        "hospital_underpayment_rate"
    ])

# Register as temp view for SQL encoding
df_filled.createOrReplaceTempView("features_raw")

# Encode categoricals using dense_rank via SQL
# This is equivalent to StringIndexer but runs in distributed SQL
df_encoded = spark.sql("""
    SELECT
        provider_id,
        drg_code,
        provider_state,

        -- Numerical features
        avg_submitted_charge,
        avg_total_payment,
        avg_medicare_payment,
        charge_to_payment_ratio,
        revenue_gap_pct,
        medicare_payment_pct,
        patient_responsibility,
        total_discharges,
        rcm_health_score,
        hospital_avg_ctp,
        hospital_underpayment_rate,
        high_volume_flag,

        -- Categorical features raw
        hospital_type,
        hospital_ownership,
        drg_severity_tier,
        rural_urban_classification,
        emergency_services,

        -- Categorical features encoded via dense_rank
        DENSE_RANK() OVER (ORDER BY hospital_type)             AS hospital_type_idx,
        DENSE_RANK() OVER (ORDER BY hospital_ownership)        AS hospital_ownership_idx,
        DENSE_RANK() OVER (ORDER BY drg_severity_tier)         AS drg_severity_tier_idx,
        DENSE_RANK() OVER (ORDER BY rural_urban_classification) AS rural_urban_classification_idx,
        DENSE_RANK() OVER (ORDER BY provider_state)            AS provider_state_idx,

        -- Target and flags
        underpayment_flag,
        denial_proxy,
        _feature_batch_id,
        _feature_date
    FROM features_raw
""")

# Define feature lists
numerical_features = [
    "avg_submitted_charge",
    "avg_total_payment",
    "avg_medicare_payment",
    "charge_to_payment_ratio",
    "revenue_gap_pct",
    "medicare_payment_pct",
    "patient_responsibility",
    "total_discharges",
    "rcm_health_score",
    "hospital_avg_ctp",
    "hospital_underpayment_rate",
    "high_volume_flag"
]

categorical_features_idx = [
    "hospital_type_idx",
    "hospital_ownership_idx",
    "drg_severity_tier_idx",
    "rural_urban_classification_idx",
    "provider_state_idx"
]

all_features = numerical_features + categorical_features_idx

print(f"Feature vector size : {len(all_features)} features")
print(f"Numerical features  : {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features_idx)}")

# Write ML feature table
df_encoded.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_ML_FEATURES)

ml_rows = spark.table(TBL_ML_FEATURES).count()
print(f"\nML feature table : {TBL_ML_FEATURES}")
print(f"Rows written     : {ml_rows:,}")

display(
    spark.table(TBL_ML_FEATURES).select(
        "provider_id",
        "drg_code",
        "charge_to_payment_ratio",
        "revenue_gap_pct",
        "hospital_type_idx",
        "drg_severity_tier_idx",
        "denial_proxy"
    ).limit(10)
)

In [0]:
# ============================================================
# STEP 4 — FEATURE STATISTICS
# ============================================================

print("=" * 55)
print("  FEATURE STATISTICS")
print("=" * 55)

spark.sql(f"""
    SELECT
        ROUND(AVG(charge_to_payment_ratio), 2) AS avg_ctp_ratio,
        ROUND(AVG(revenue_gap_pct), 2)          AS avg_revenue_gap_pct,
        ROUND(AVG(medicare_payment_pct), 2)     AS avg_medicare_pct,
        ROUND(AVG(total_discharges), 0)         AS avg_discharges,
        ROUND(AVG(rcm_health_score), 1)         AS avg_rcm_score,
        SUM(denial_proxy)                       AS total_denial_proxy_1,
        COUNT(*)                                AS total_rows,
        ROUND(SUM(denial_proxy)/COUNT(*)*100, 2) AS denial_proxy_rate_pct
    FROM {TBL_ML_FEATURES}
""").show(truncate=False)

print("Denial proxy by DRG severity tier:")
spark.sql(f"""
    SELECT
        drg_severity_tier,
        COUNT(*)                                  AS rows,
        SUM(denial_proxy)                         AS denial_count,
        ROUND(SUM(denial_proxy)/COUNT(*)*100, 2)  AS denial_rate_pct
    FROM {TBL_ML_FEATURES}
    GROUP BY drg_severity_tier
    ORDER BY denial_rate_pct DESC
""").show(truncate=False)

print("Denial proxy by hospital type:")
spark.sql(f"""
    SELECT
        hospital_type,
        COUNT(*)                                  AS rows,
        SUM(denial_proxy)                         AS denial_count,
        ROUND(SUM(denial_proxy)/COUNT(*)*100, 2)  AS denial_rate_pct
    FROM {TBL_ML_FEATURES}
    GROUP BY hospital_type
    ORDER BY denial_rate_pct DESC
""").show(truncate=False)

In [0]:
# ============================================================
# STEP 5 — LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "ml",
    status           = "SUCCESS",
    rows_read        = feature_count,
    rows_written     = ml_rows,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"features: {ml_rows:,} rows | "
        f"{len(all_features)} features engineered | "
        f"denial proxy rate: {denial_rate:.2f}% | "
        f"CTP 75th percentile threshold: {ctp_75th:.4f}"
    )
)